### Tools 

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

        1. A schema, including the name of the tool, a description, and/or argument definitions         (often a JSON schema)
        2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.1-8b-instant")
response=model.invoke("Hello, how are you?")
print(response.content)

I'm functioning properly, thank you for asking. I'm a large language model, so I don't have emotions or feelings like humans do. I'm here to assist you with any questions or tasks you may have, though. How can I help you today?


In [3]:
## Tools

from langchain.tools import  tool 
@tool # this is a decorator that will convert the function to a tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"The weather at {location} is sunny"

model_with_tools=model.bind_tools([get_weather]) # Bind tools with the model

In [4]:
response=model_with_tools.invoke("What is the weather in karachi?")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'pvygee6tn', 'function': {'arguments': '{"location":"Karachi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.027079594, 'completion_tokens_details': None, 'prompt_time': 0.013117749, 'prompt_tokens_details': None, 'queue_time': 0.057898451, 'total_time': 0.040197343}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cf583-6661-7953-a9c1-146636ca1bdc-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Karachi'}, 'id': 'pvygee6tn', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}


In [5]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Karachi'},
  'id': 'pvygee6tn',
  'type': 'tool_call'}]

### Tool Execution Loop

In [6]:
#Step 1: Model generates tool calls
messages=[{"role": "user", "content": "What is the weather like in San Francisco?"}] 
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the genrated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)

However, the weather at San Francisco might be subject to change. The provided weather information might not be accurate or up-to-date.


In [7]:
messages

[{'role': 'user', 'content': 'What is the weather like in San Francisco?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1aw9pn7sv', 'function': {'arguments': '{"location":"San Francisco"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 221, 'total_tokens': 236, 'completion_time': 0.026165597, 'completion_tokens_details': None, 'prompt_time': 0.015165928, 'prompt_tokens_details': None, 'queue_time': 0.068427252, 'total_time': 0.041331525}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf592-09c8-7791-99da-63beff159714-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'San Francisco'}, 'id': '1aw9pn7sv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 15, 'total_tokens': 236}),
 T

In [8]:
model_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001894C8529E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001894C8528F0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])